# 00 — Environment Setup & GPU Check

Run this notebook first after creating your OpenShift AI Workbench.
It verifies GPU access, installs dependencies, and runs a smoke test.

**OpenShift AI Workbench settings:**
- Image: PyTorch (CUDA)
- Container size: Large (8+ CPU, 32+ GB RAM)
- Accelerator: NVIDIA GPU x1
- Persistent storage: >= 30 GB

## 1. GPU Verification

In [1]:
!nvidia-smi

Tue Aug 11 05:49:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:10:1C.0 Off |                    0 |
| N/A   34C    P0             61W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
 %pip install vllm==0.18.0

INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of nvidia-cutlass-dsl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of nvidia-cutlass-dsl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 90.6 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 179.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 434.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 155.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 156.3 MB

In [2]:
%pip show torch

# if torch is not already provided:
#!pip install torch==2.10.0

Name: torch
Version: 2.10.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /opt/app-root/lib64/python3.12/site-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: compressed-tensors, flash_attn, flashinfer-python, quack-kernels, torch_c_dlpack_ext, torchaudio, torchvision, vllm, xgrammar
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip show flash_attn

# if flash_attn is not already provided:
#!pip install flash_attn==2.8.3

Name: flash_attn
Version: 2.8.3
Summary: Flash Attention: Fast and Memory-Efficient Exact Attention
Home-page: https://github.com/Dao-AILab/flash-attention
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /opt/app-root/lib64/python3.12/site-packages
Requires: einops, torch
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:             {gpu_name}")
    print(f"VRAM:            {vram_gb:.1f} GB")

    if vram_gb < 20:
        print("\n⚠  VRAM < 20 GB — Qwen3-8B in fp16 may not fit.")
        print("   The experiment notebooks will use 4-bit quantization (BitsAndBytesConfig).")
    else:
        print("\n✓  Enough VRAM for Qwen3-8B in fp16.")
else:
    raise RuntimeError(
        "No CUDA GPU detected. Make sure your Workbench has a GPU accelerator assigned."
    )

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             NVIDIA A100-SXM4-40GB
VRAM:            42.4 GB

✓  Enough VRAM for Qwen3-8B in fp16.


## 2. Environment & Dependencies

Set `HF_HOME` to persistent storage so downloads survive pod restarts.
Re-run the cell below if the pod was recreated. Pip will skip already-installed packages.

In [5]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

%pip install git+https://github.com/NVIDIA/kvpress.git@v0.5.4 --no-deps
%pip install vllm==0.18.0 transformers==4.57.6 datasets==3.6.0 \
    rouge-score==0.1.2 matplotlib==3.10.8 pandas==2.3.3 \
    bitsandbytes==0.49.2 accelerate==1.12.0 fire==0.7.1

# kvpress evaluation framework dependencies (uv sync --extra eval)
%pip install rouge==1.0.1 jieba==0.42.1 fuzzywuzzy==0.18.0 bert-score==0.3.13 \
    --index-url https://pypi.org/simple/

  Cloning https://github.com/NVIDIA/kvpress.git (to revision v0.5.4) to /tmp/pip-req-build-kcoerw_0
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/kvpress.git /tmp/pip-req-build-kcoerw_0
  Running command git checkout -q 6d965557a5b9f0201a2301b23c454473dd681d0d
  Resolved https://github.com/NVIDIA/kvpress.git to commit 6d965557a5b9f0201a2301b23c454473dd681d0d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kvpress: filename=kvpress-0.5.4-py3-none-any.whl size=110004 sha256=2c888d3c2c45ad68da5a6a5ca323f9b0cc8472dba5359e62c05b6e20af1a114a
  Stored in directory: /tmp/pip-ephem-wheel-cache-u6fp3row/wheels/e5/d0/d2/f5db0334cb0eb1c1d563687c65cd1b9282dd6c62f5c6616ef7
Successfully built kvpress
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 2

## 3. Verify Imports

In [6]:
from importlib.metadata import version
import kvpress, vllm

for pkg in ["kvpress", "vllm", "transformers", "datasets", "rouge-score", "matplotlib", "pandas"]:
    print(f"{pkg:20s} {version(pkg)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kvpress              0.5.4
vllm                 0.18.0
transformers         4.57.6
datasets             3.6.0
rouge-score          0.1.2
matplotlib           3.10.8
pandas               2.3.3


## 4. Verify kvpress Press Types

In [7]:
from kvpress import KeyDiffPress, BlockPress, PrefillDecodingPress, CompressionRatioDecodingPress

press = PrefillDecodingPress(
    prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128),
    decoding_press=CompressionRatioDecodingPress(
        base_press=KeyDiffPress(), target_compression_ratio=0.5,
    ),
)

print(f"PrefillDecodingPress: {press}")
print(f"  prefilling: {press.prefilling_press}")
print(f"  decoding:   {press.decoding_press}")
print("\n✓  Press types instantiated successfully.")

PrefillDecodingPress: PrefillDecodingPress(prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128), decoding_press=CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5))
  prefilling: BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128)
  decoding:   CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5)

✓  Press types instantiated successfully.


## 5. Smoke Test — CUDA Tensor Round-Trip

In [12]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
y = x @ x.T
print(f"Matrix multiply on GPU OK — result shape: {y.shape}, dtype: {y.dtype}")
del x, y
torch.cuda.empty_cache()

Matrix multiply on GPU OK — result shape: torch.Size([1024, 1024]), dtype: torch.float16


## 6. Dataset Access Check

In [8]:
from datasets import load_dataset

ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
print(f"Paul Graham essays loaded: {len(ds)} rows")
print(f"Columns: {ds.column_names}")
print(f"Context length: {len(ds[0]['context'].split())} words")
print(f"Needle: {ds[0]['needle'][:80]}...")
print(f"Question: {ds[0]['question']}")

Paul Graham essays loaded: 1 rows
Columns: ['context', 'needle', 'question', 'answer_prefix', 'max_new_tokens']
Context length: 525450 words
Needle: 

Remember, the best thing to do in San Francisco is eat a sandwich and sit in D...
Question: 

Question: Based on the content of the book, what is the best thing to do in San Francisco?


## Done

If all cells above ran without errors, your environment is ready.  
Proceed to:
- `01_kvpress_fork_setup.ipynb` — install kvpress from fork (if needed)
- `02_kvpress_niah.ipynb` — KeyDiffPress NIAH experiments
- `03_vllm_fork_setup.ipynb` — install vLLM from fork (if needed)
- `04_vllm_niah.ipynb` — vLLM baseline NIAH experiments